In [ ]:
# CELL 1 - Setup
!pip install -q transformers datasets accelerate scikit-learn pandas numpy sentencepiece protobuf

import os, re, json, math, random, subprocess
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    subprocess.run(['nvidia-smi'], check=False)

PROJECT = '/content/DePlag_Stable_90Plus' if os.path.exists('/content') else '/kaggle/working/DePlag_Stable_90Plus'
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
for d in ['data', 'models', 'reports', 'evaluation']:
    os.makedirs(d, exist_ok=True)
print('Working directory:', os.getcwd())
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# CELL 2 - Configuration (94%+ target: roberta-large + lower LR + longer seq)
MODEL_NAME = 'cross-encoder/stsb-roberta-large'
MAX_LENGTH = 256
EPOCHS = 3
LR = 2e-6
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
WARMUP_RATIO = 0.06
USE_MRPC = True
USE_PAWS = True
USE_QQP = True
USE_STSB = True
MAX_TRAIN_EXAMPLES = 120000
MAX_EVAL_EXAMPLES = 10000
print({'model': MODEL_NAME, 'max_length': MAX_LENGTH, 'epochs': EPOCHS, 'lr': LR})

In [ ]:
# CELL 3 - Dataset loading
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split

def clean_text(text):
    if not isinstance(text, str): return ''
    return ' '.join(text.strip().split())

def norm_key(text):
    text = clean_text(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return ' '.join(text.split())

def as_pair_df(rows, source):
    df = pd.DataFrame(rows)
    df['text_a'] = df['text_a'].map(clean_text)
    df['text_b'] = df['text_b'].map(clean_text)
    df = df[(df.text_a != '') & (df.text_b != '')].copy()
    df['label'] = df['label'].astype(int)
    df = df[df.label.isin([0, 1])].copy()
    df['source'] = source
    return df[['text_a', 'text_b', 'label', 'source']].reset_index(drop=True)

def split_val_test(df, test_size=0.5):
    val, test = train_test_split(df, test_size=test_size, stratify=df.label, random_state=SEED)
    return val.reset_index(drop=True), test.reset_index(drop=True)

train_parts, val_parts, test_parts = [], [], []

if USE_MRPC:
    mrpc = load_dataset('glue', 'mrpc')
    def mrpc_rows(split):
        return [{'text_a': x['sentence1'], 'text_b': x['sentence2'], 'label': x['label']} for x in mrpc[split]]
    train_parts.append(as_pair_df(mrpc_rows('train'), 'mrpc'))
    mrpc_val, mrpc_test = split_val_test(as_pair_df(mrpc_rows('validation'), 'mrpc'))
    val_parts.append(mrpc_val)
    test_parts.append(mrpc_test)

if USE_PAWS:
    paws = load_dataset('paws', 'labeled_final')
    def paws_rows(split):
        return [{'text_a': x['sentence1'], 'text_b': x['sentence2'], 'label': x['label']} for x in paws[split]]
    train_parts.append(as_pair_df(paws_rows('train'), 'paws'))
    val_parts.append(as_pair_df(paws_rows('validation'), 'paws'))
    test_parts.append(as_pair_df(paws_rows('test'), 'paws'))

if USE_QQP:
    qqp = load_dataset('glue', 'qqp')
    def qqp_rows(split):
        rows = []
        for x in qqp[split]:
            if x['question1'] and x['question2']:
                rows.append({'text_a': x['question1'], 'text_b': x['question2'], 'label': x['label']})
        return rows
    train_parts.append(as_pair_df(qqp_rows('train'), 'qqp'))
    qqp_val, qqp_test = split_val_test(as_pair_df(qqp_rows('validation'), 'qqp'))
    val_parts.append(qqp_val)
    test_parts.append(qqp_test)

if USE_STSB:
    try:
        stsb = load_dataset('mteb/stsbenchmark-sts')
        score_key, s1_key, s2_key = 'score', 'sentence1', 'sentence2'
    except Exception:
        stsb = load_dataset('stsb_multi_mt', 'en')
        score_key, s1_key, s2_key = 'similarity_score', 'sentence1', 'sentence2'
    def stsb_rows_clean(split):
        rows = []
        for x in stsb[split]:
            score = float(x[score_key])
            if score >= 3.5:
                label = 1
            elif score <= 2.5:
                label = 0
            else:
                continue
            rows.append({'text_a': x[s1_key], 'text_b': x[s2_key], 'label': label})
        return rows
    train_parts.append(as_pair_df(stsb_rows_clean('train'), 'stsb'))
    val_parts.append(as_pair_df(stsb_rows_clean('validation'), 'stsb'))
    test_parts.append(as_pair_df(stsb_rows_clean('test'), 'stsb'))

train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print('Raw sizes:', {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})

In [ ]:
# CELL 4 - Dedupe, leakage protection, balance
def add_pair_key(df):
    a = df.text_a.map(norm_key)
    b = df.text_b.map(norm_key)
    df = df.copy()
    df['pair_key'] = np.minimum(a, b) + ' ||| ' + np.maximum(a, b)
    return df.drop_duplicates('pair_key').reset_index(drop=True)

def balance_binary(df, max_examples=None, name='split'):
    counts = df.label.value_counts().to_dict()
    if 0 not in counts or 1 not in counts:
        return df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    per_class = min(counts[0], counts[1])
    if max_examples is not None:
        per_class = min(per_class, max_examples // 2)
    out = pd.concat([
        df[df.label == 0].sample(per_class, random_state=SEED),
        df[df.label == 1].sample(per_class, random_state=SEED),
    ], ignore_index=True)
    return out.sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df = add_pair_key(train_df)
val_df = add_pair_key(val_df)
test_df = add_pair_key(test_df)
train_keys = set(train_df.pair_key)
val_df = val_df[~val_df.pair_key.isin(train_keys)].reset_index(drop=True)
val_keys = set(val_df.pair_key)
test_df = test_df[~test_df.pair_key.isin(train_keys | val_keys)].reset_index(drop=True)
train_df = balance_binary(train_df, MAX_TRAIN_EXAMPLES, 'train')
val_df = balance_binary(val_df, MAX_EVAL_EXAMPLES, 'validation')
test_df = balance_binary(test_df, MAX_EVAL_EXAMPLES, 'test')

for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print('\n' + name, len(df))
    print(df.source.value_counts())
    print(df.label.value_counts(normalize=True))

train_df.drop(columns=['pair_key']).to_csv('data/train_pairs.csv', index=False)
val_df.drop(columns=['pair_key']).to_csv('data/val_pairs.csv', index=False)
test_df.drop(columns=['pair_key']).to_csv('data/test_pairs.csv', index=False)

In [ ]:
# CELL 5 - Tokenize
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def to_dataset(df):
    return Dataset.from_pandas(df[['text_a', 'text_b', 'label', 'source']].reset_index(drop=True), preserve_index=False)

raw_ds = DatasetDict({
    'train': to_dataset(train_df),
    'validation': to_dataset(val_df),
    'test': to_dataset(test_df),
})

def tokenize_batch(batch):
    return tokenizer(batch['text_a'], batch['text_b'], truncation=True, max_length=MAX_LENGTH)

encoded = raw_ds.map(tokenize_batch, batched=True, remove_columns=['text_a', 'text_b', 'source'])
encoded = encoded.rename_column('label', 'labels')
print(encoded)

In [ ]:
# CELL 6 - Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, classification_report, confusion_matrix

def logits_to_proba(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = np.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
    if logits.ndim == 1 or logits.shape[1] == 1:
        z = np.clip(logits.reshape(-1), -50, 50)
        return 1.0 / (1.0 + np.exp(-z))
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(np.clip(logits, -50, 50))
    return exp_logits[:, 1] / np.clip(exp_logits.sum(axis=1), 1e-12, None)

def metric_dict(y_true, proba, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba, dtype=np.float64)
    pred = (proba >= threshold).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, pred)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, pred)),
        'roc_auc': float(roc_auc_score(y_true, proba)) if len(set(y_true)) == 2 else 0.0,
        'threshold': float(threshold),
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metric_dict(labels, logits_to_proba(logits), threshold=0.5)

def threshold_sweep(y_true, proba):
    rows = []
    for t in np.linspace(0.05, 0.95, 181):
        rows.append(metric_dict(y_true, proba, t))
    return pd.DataFrame(rows).sort_values(['accuracy', 'mcc'], ascending=False)

In [ ]:
# CELL 7 - Train cross-encoder (num_labels=2, 94%+ config: roberta-large, fp32)
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True)
model.config.use_cache = False
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

collator = DataCollatorWithPadding(tokenizer=tokenizer)
steps_per_epoch = max(1, math.ceil(len(encoded['train']) / TRAIN_BATCH_SIZE))
warmup_steps = int(steps_per_epoch * EPOCHS * WARMUP_RATIO)
major_cc = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
USE_BF16 = bool(torch.cuda.is_available() and major_cc >= 8)
USE_FP16 = False  # Disabled: large model + reinitialized head is unstable under FP16

print({'steps_per_epoch': steps_per_epoch, 'warmup_steps': warmup_steps, 'fp16': USE_FP16, 'bf16': USE_BF16})

args = TrainingArguments(
    output_dir='models/cross_encoder',
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    gradient_accumulation_steps=2,  # effective batch = 32
    warmup_steps=warmup_steps,
    max_grad_norm=1.0,
    optim='adamw_torch',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    logging_nan_inf_filter=True,
    save_total_limit=2,
    label_smoothing_factor=0.05,
    fp16=USE_FP16,
    bf16=USE_BF16,
    report_to='none',
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded['train'],
    eval_dataset=encoded['validation'],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
trainer.save_model('models/best_cross_encoder')
tokenizer.save_pretrained('models/best_cross_encoder')

In [ ]:
# CELL 8 - Threshold tuning and final evaluation
def predict_proba(split_name):
    pred = trainer.predict(encoded[split_name])
    labels = np.asarray(pred.label_ids).astype(int)
    proba = logits_to_proba(pred.predictions)
    print(split_name, {'n': len(labels), 'finite': bool(np.isfinite(proba).all()), 'min': float(proba.min()), 'max': float(proba.max())})
    return labels, proba

y_val, val_proba = predict_proba('validation')
y_test, test_proba = predict_proba('test')

val_sweep = threshold_sweep(y_val, val_proba)
display(val_sweep.head(15))
best_threshold = float(val_sweep.iloc[0].threshold)
test_pred = (test_proba >= best_threshold).astype(int)

val_metrics = metric_dict(y_val, val_proba, best_threshold)
test_metrics = metric_dict(y_test, test_proba, best_threshold)

print('Best threshold:', best_threshold)
print('Validation metrics:', json.dumps(val_metrics, indent=2))
print('Test metrics:', json.dumps(test_metrics, indent=2))
print(classification_report(y_test, test_pred, target_names=['Not Plagiarism', 'Plagiarism'], zero_division=0))

source_rows = []
for src in sorted(test_df.source.unique()):
    idx = test_df.source.values == src
    m = metric_dict(y_test[idx], test_proba[idx], best_threshold)
    m['source'] = src
    m['n'] = int(idx.sum())
    source_rows.append(m)
    print(src, m)

results = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'learning_rate': LR,
    'threshold': best_threshold,
    'train_examples': len(train_df),
    'validation_examples': len(val_df),
    'test_examples': len(test_df),
    'validation': val_metrics,
    'test': test_metrics,
    'per_source': source_rows,
    'confusion_matrix': confusion_matrix(y_test, test_pred).tolist(),
}

with open('reports/results.json', 'w') as f:
    json.dump(results, f, indent=2)
pd.DataFrame(source_rows).to_csv('reports/per_source_metrics.csv', index=False)
pd.DataFrame({'label': y_test, 'proba': test_proba, 'pred': test_pred, 'source': test_df.source.values}).to_csv('reports/test_predictions.csv', index=False)
print('Saved reports/results.json, reports/per_source_metrics.csv, reports/test_predictions.csv')